# 02 — Fine-tuning Baseline: Demonstrating Catastrophic Forgetting

The simplest approach to sequential task learning is **naive fine-tuning**: train on each task
sequentially with no mechanism to preserve knowledge of previous tasks.

This notebook demonstrates catastrophic forgetting — the tendency of neural networks to abruptly
lose previously learned knowledge when trained on new data.  The results here serve as the
lower-bound baseline against which all other methods are compared.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocess import get_split_cifar100
from src.models.base_model import get_model
from src.evaluate import evaluate_task, build_accuracy_matrix, print_metrics
from src.train import run_continual

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Run Fine-tuning

We train ResNet-18 on all 20 Split-CIFAR-100 tasks sequentially.
After each task we record the accuracy on *all previous tasks*.

In [ ]:
R_ft = run_continual(
    method='finetune',
    dataset='cifar100',
    n_tasks=20,
    epochs_per_task=10,
    lr=0.1,
    batch_size=128,
    device=device,
    seed=42,
    save_dir='../results/baseline',
    verbose=True,
)

## 2. Accuracy Matrix

In [ ]:
T = R_ft.shape[0]
fig, ax = plt.subplots(figsize=(11, 9))
mask = np.isnan(R_ft)
display = np.where(mask, 0.0, R_ft * 100)
im = ax.imshow(display, vmin=0, vmax=100, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, label='Accuracy (%)')
ax.set_xticks(range(T))
ax.set_yticks(range(T))
ax.set_xticklabels([f'T{j}' for j in range(T)], fontsize=8)
ax.set_yticklabels([f'T{i}' for i in range(T)], fontsize=8)
ax.set_xlabel('Task evaluated', fontsize=12)
ax.set_ylabel('After training task', fontsize=12)
ax.set_title('Fine-tuning Baseline — Accuracy Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/baseline/acc_matrix_finetune.png', bbox_inches='tight')
plt.show()

## 3. Forgetting Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: accuracy on task 0 as more tasks are trained
task0_curve = [R_ft[i, 0] * 100 for i in range(T)]
axes[0].plot(range(T), task0_curve, marker='o', color='#e74c3c', linewidth=2)
axes[0].fill_between(range(T), task0_curve, alpha=0.2, color='#e74c3c')
axes[0].set_xlabel('Tasks trained', fontsize=12)
axes[0].set_ylabel('Accuracy on Task 0 (%)', fontsize=12)
axes[0].set_title('Catastrophic Forgetting on Task 0', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 105)

# Right: accuracy on every past task after all 20 tasks
final_row = R_ft[T - 1, :]
axes[1].bar(range(T), final_row * 100, color='#e74c3c', alpha=0.75)
axes[1].set_xlabel('Task index', fontsize=12)
axes[1].set_ylabel('Final accuracy (%)', fontsize=12)
axes[1].set_title('Per-task Accuracy After All 20 Tasks', fontsize=12, fontweight='bold')
axes[1].set_ylim(0, 105)
axes[1].grid(True, axis='y', alpha=0.3)

plt.suptitle('Fine-tuning: Catastrophic Forgetting', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../results/baseline/forgetting_finetune.png', bbox_inches='tight')
plt.show()

## 4. Metrics Summary

In [ ]:
print_metrics(R_ft, method='Fine-tuning Baseline')

## Observation

The accuracy on early tasks collapses to near zero as subsequent tasks overwrite shared weights.
The large negative BWT confirms severe catastrophic forgetting — the fundamental problem this
project addresses through causal-guided pruning.